# Customer Churn Analysis & Machine Learning Notebook

**Project:** Customer Churn Prediction System  
**Dataset:** Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)  
**Goal:** Analyze customer demographics and subscription patterns, build preprocessing pipelines, train 4 classification algorithms, evaluate benchmark metrics, and select the optimal model based on ROC-AUC.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

%matplotlib inline
sns.set_theme(style='darkgrid')

## 1. Load and Inspect Dataset

In [ ]:
df_raw = pd.read_csv('../dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head()

## 2. Data Cleaning & Preprocessing

In [ ]:
# Drop customerID identifier
df = df_raw.drop(columns=['customerID'])

# Convert TotalCharges to numeric, coercing invalid/blank values to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# Encode Target Variable Churn: Yes -> 1, No -> 0
df['Churn_Binary'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f'Cleaned dataset shape: {df.shape}')
print('Target Churn breakdown:')
print(df['Churn'].value_counts(normalize=True) * 100)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Churn by Contract Type
sns.countplot(data=df, x='Contract', hue='Churn', palette='viridis', ax=axes[0, 0])
axes[0, 0].set_title('Churn Rate by Contract Type')

# Churn by Internet Service
sns.countplot(data=df, x='InternetService', hue='Churn', palette='magma', ax=axes[0, 1])
axes[0, 1].set_title('Churn Rate by Internet Service Provider')

# Churn by Payment Method
sns.countplot(data=df, x='PaymentMethod', hue='Churn', palette='rocket', ax=axes[1, 0])
axes[1, 0].set_title('Churn Rate by Payment Method')
axes[1, 0].tick_params(axis='x', rotation=30)

# Monthly Charges Distribution by Churn
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, palette='crest', ax=axes[1, 1])
axes[1, 1].set_title('Monthly Charges Density Distribution')

plt.tight_layout()
plt.show()

## 4. Machine Learning Model Pipeline & Training

In [ ]:
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

X = df[num_cols + cat_cols]
y = df['Churn_Binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
plt.figure(figsize=(9, 7))

for name, clf in models.items():
    pipeline = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Algorithm': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': roc_auc
    })
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Benchmark Comparison')
plt.legend(loc='lower right')
plt.show()

results_df = pd.DataFrame(results)
results_df